In [ ]:
if 'google.colab' in str(get_ipython()):
  !pip install git+https://github.com/FullControlXYZ/fullcontrol --quiet
import lab.fullcontrol.infaxis as fci
import fullcontrol as fc
from math import sin, cos, tau

fc.Point = fci.Point
fc.GcodeControls = fci.GcodeControls
fc.transform = fci.transform
fc.Axis = fci.Axis

In [ ]:
EW = 0.6
EH = 0.3

print_settings = {'extrusion_width': EW,'extrusion_height': EH}
gcode_controls = fc.GcodeControls(
    head_chain = [fc.Axis(name='B')],
    bed_chain = [fc.Axis(name='C')],
    distance_axis = True,
    verbose = True,
    initialization_data=print_settings 
    )

In [ ]:
def vase_from_trace(trace_points,density):
    steps = []
    for i in range(len(trace_points)-1):
        p = trace_points[i]
        p_next = trace_points[i+1]
        r = (p.x**2 + p.y**2)**0.5
        r_next = (p_next.x**2 + p_next.y**2)**0.5
        dr = r_next - r
        dz = p_next.z-p.z
        ptilt = p.axes['B']
        dtilt = p_next.axes['B']-ptilt
            
        dc = p_next.axes['C']-p.axes['C']
        for j in range(density):
            angle = p.axes['C'] + dc * j / density
            tilt = ptilt + dtilt * j / density
            r_final = r + dr * j / density
            z_final = p.z + dz * j / density
   
            steps.append(fc.Point(x=r_final*sin(angle/360*tau), y=r_final*cos(angle/360*tau), z=z_final, axes={'B':tilt,'C':angle}))
    
    return steps

density = 360
r_start = 10
r_tilt = 5 # 10 layers
tilt_start = 0
tilt_end = 90
h = EH # height of each layer
z_start = h*0.5 # starting z height
layers = 20 # number of layers in the z direction for first segment
arc_layers = int((r_tilt * (tilt_end - tilt_start) / 360 *tau)/h)
d_tilted = 5
layers_tilted = int(d_tilted/h)

trace = []

for i in range(layers):
    r = r_start
    tilt = tilt_start
    angle = i * 360
    z = z_start + h * i

    trace.append(fc.Point(x=r, y=0, z=z, axes={'B':tilt,'C':angle}))

angle_offset = fc.last_point(trace).axes['C']
z_offset = fc.last_point(trace).z

for i in range(1,arc_layers):
    tilt = tilt_start + (tilt_end - tilt_start) * i / (arc_layers - 1)
    r = r_start+r_tilt*(1-cos(tilt*tau/360))
    z = z_offset+r_tilt*(sin(tilt*tau/360))
    angle = i * 360 + angle_offset
    trace.append(fc.Point(x=r, y=0, z=z, axes={'B':-tilt,'C':angle}))

angle_offset = fc.last_point(trace).axes['C']
z_offset = fc.last_point(trace).z
r_offset = fc.last_point(trace).x

for i in range(1,layers_tilted):
    tilt = tilt_end
    r = r_offset + d_tilted * i / (layers_tilted - 1) * (sin(tilt*tau/360))
    z = z_offset + d_tilted * i / (layers_tilted - 1) * cos(tilt*tau/360)
    angle = i * 360 + angle_offset
    trace.append(fc.Point(x=0, y=r, z=z, axes={'B':-tilt,'C':angle}))

steps = vase_from_trace(trace, density)
steps.append(fc.last_point(trace))

for step in steps:
    if type(step).__name__ == 'Point':
        # color is a gradient from A=0 (blue) to A=90 (red)
        step.color = [((abs(step.axes['B']))/90), 0, 1-((abs(step.axes['B']))/90)]

fig = fc.transform(steps, 'fig', fc.PlotControls(color_type='manual',style='tube', zoom=0.75), show_tips=False)
fig.show()

gcode = fc.transform(steps,'gcode',gcode_controls)
print('first ten gcode lines:\n' + '\n'.join(gcode.split('\n')[:10]))
print('')
print('final ten gcode lines:\n' + '\n'.join(gcode.split('\n')[-10:]))